# 1.6 Bonus - The open-source Spark Declarative Pipelines engine
Databricks donated Declarative Pipelines to Apache Spark. Since **Spark 4.1** the open-source distribution ships `pyspark.pipelines` and the `spark-pipelines` CLI. This notebook runs a pipeline on that **real engine**, not on the `adelab` emulation, so you can see exactly which parts of the API are open source.

| Feature | Apache Spark 4.2 OSS | Databricks (Lakeflow SDP) |
|---|---|---|
| `@dp.table`, `@dp.materialized_view`, `@dp.temporary_view`, `dp.create_streaming_table` | ✅ | ✅ |
| `@dp.append_flow` (multi-flow), `dp.create_sink` | ✅ | ✅ |
| SQL `CREATE MATERIALIZED VIEW / STREAMING TABLE` | ✅ | ✅ |
| Expectations (`@dp.expect*`) | ❌ | ✅ |
| `cluster_by` (Liquid Clustering) | ❌ (Delta) | ✅ |
| AUTO CDC into Delta | ❌ (SCD1 only, needs a row-level-capable connector) | ✅ SCD1 + SCD2 |
| Event log, UI, serverless, incremental MV refresh | ❌ | ✅ |

![Capability boundary: the Apache Spark 4.1+ open-source core (tables, materialized and temporary views, append flows, sinks, SQL definitions, the spark-pipelines CLI, flow lifecycle and checkpoints) sits inside Databricks Lakeflow SDP, which adds expectations, cluster_by, AUTO CDC SCD1 and SCD2, the event log, the pipeline UI, serverless and Unity Catalog](../images/1.6-oss-vs-databricks.svg)

**How to read it:** the inner box is everything `ingest.py` and `gold.sql` below use, so they run on both engines. Anything in the outer ring (for example `@dp.expect`) only runs on Databricks; the "Try it" section at the end shows the failure.

On Windows, `spark-pipelines` is a bash script. `adelab.oss_sdp.run()` starts the same Java entry point (`org.apache.spark.deploy.SparkPipelines`) with a generated `spark-pipeline.yml`.

In [ ]:
from adelab.classroom import setup
from adelab import datagen, oss_sdp

DA = setup("1.6", reset=True)
for flow in ("a", "b"):
    for _ in range(2):
        datagen.land_events(f"{DA.paths.landing}/{flow}", flow)

## The project
A Python file plus a **SQL** file. The Python file uses only the open-source subset of the API. The SQL file defines a materialized view over a Python-defined table.

In [ ]:
DA.show_source("oss-spark-pipelines")

Configuration values are read with `spark.conf.get(...)`. The OSS CLI passes them as Spark confs, so the keys are prefixed with `spark.`.

In [ ]:
CONF = {
    "spark.adelab.source_a": f"{DA.paths.landing}/a",
    "spark.adelab.source_b": f"{DA.paths.landing}/b",
    "spark.adelab.sink_path": f"{DA.paths.working_dir}/exports/error_events",
}

## `spark-pipelines dry-run`
Validates the graph (resolves every dataset and flow) without running anything.

In [ ]:
oss_sdp.run(DA.course_dir / "oss-spark-pipelines", "oss_multi_flow", CONF,
            command="dry-run")

## `spark-pipelines run --full-refresh-all`
Watch the flow lifecycle in the log: `QUEUED → PLANNING → STARTING → RUNNING → COMPLETED`. It is the same state machine the Databricks pipeline UI shows.

![Flow lifecycle: QUEUED, PLANNING, STARTING, RUNNING, then COMPLETED with the checkpoint committed, or FAILED with the batch not committed and downstream flows skipped](../images/1.6-flow-lifecycle.svg)

**How to read it:** every flow in the log below (`flow_a`, `flow_b`, `silver_events`, `gold_daily_status`, `export_errors`) walks this path. A flow starts only after the datasets it reads are done, so the log interleaves the states of different flows.

In [ ]:
oss_sdp.run(DA.course_dir / "oss-spark-pipelines", "oss_multi_flow", CONF, full_refresh=True)

## Read the results from this notebook
The OSS engine ran in a separate Spark application with its own catalog. Read its Delta tables by path.

In [ ]:
for t in ("bronze_events", "silver_events", "gold_daily_status"):
    df = spark.read.format("delta").load(oss_sdp.table_path("oss_multi_flow", t))
    print(f"{t:<20} {df.count():>5} rows")
display(spark.read.format("delta").load(oss_sdp.table_path("oss_multi_flow", "gold_daily_status"))
        .orderBy("event_date", "status"))
display(spark.read.format("delta").load(CONF["spark.adelab.sink_path"]).groupBy("source_flow").count())

## Incremental run
Land one more file and run **without** a full refresh. Only the new file is read.

In [ ]:
datagen.land_events(f"{DA.paths.landing}/a", "a")
oss_sdp.run(DA.course_dir / "oss-spark-pipelines", "oss_multi_flow", CONF)
print(spark.read.format("delta").load(oss_sdp.table_path("oss_multi_flow", "bronze_events")).count(), "bronze rows")

## Try it
- Add `@dp.expect("x", "amount > 0")` to `silver_events` and run again. The OSS engine fails with an `AttributeError`, because expectations are not in open-source Spark yet.
- Copy `ingest.py` into a Databricks pipeline. It runs unchanged.

In [ ]:
DA.cleanup()